# PNC Banking Project - Quick Setup Guide

## 📋 Pre-requisites Checklist

Before starting, ensure you have:
- [ ] Python 3.8 or higher installed
- [ ] PostgreSQL 12 or higher installed and running
- [ ] VS Code (or any code editor)
- [ ] Terminal/Command Prompt access

## 🚀 Step-by-Step Setup Instructions

### Step 1: Create Project Directory Structure

```bash
mkdir pnc_churn_analysis
cd pnc_churn_analysis

# Create subdirectories
mkdir sql
mkdir src
```

### Step 2: Create All Project Files

Copy the content from each artifact into the corresponding files:

1. **README.md** → Root directory
2. **.gitignore** → Root directory
3. **requirements.txt** → Root directory
4. **schema.sql** → `sql/` directory
5. **1_generate_data.py** → `src/` directory
6. **2_churn_analysis_and_modeling.ipynb** → `src/` directory

Your structure should look like:
```
pnc_churn_analysis/
├── .gitignore
├── README.md
├── requirements.txt
├── sql/
│   └── schema.sql
└── src/
    ├── 1_generate_data.py
    └── 2_churn_analysis_and_modeling.ipynb
```

### Step 3: Set Up Python Environment

```bash
# Create virtual environment
python -m venv .venv

# Activate it
# On Windows:
.venv\Scripts\activate

# On Mac/Linux:
source .venv/bin/activate

# Install dependencies
pip install -r requirements.txt
```

### Step 4: Set Up PostgreSQL Database

#### Option A: Using Command Line

```bash
# Create database
createdb pnc_banking

# Run schema
psql -d pnc_banking -f sql/schema.sql
```

#### Option B: Using pgAdmin or Another GUI

1. Open pgAdmin
2. Create a new database named `pnc_banking`
3. Open Query Tool
4. Copy content from `sql/schema.sql`
5. Execute the script

### Step 5: Update Database Credentials

Update the `DB_PARAMS` dictionary in **two files**:

**File 1:** `src/1_generate_data.py` (Line 22-27)
```python
DB_PARAMS = {
    'host': 'localhost',
    'database': 'pnc_banking',
    'user': 'YOUR_USERNAME',      # ← Change this
    'password': 'YOUR_PASSWORD'    # ← Change this
}
```

**File 2:** `src/2_churn_analysis_and_modeling.ipynb` (Cell 2)
```python
DB_PARAMS = {
    'host': 'localhost',
    'database': 'pnc_banking',
    'user': 'YOUR_USERNAME',      # ← Change this
    'password': 'YOUR_PASSWORD'    # ← Change this
}
```

### Step 6: Generate and Load Data

```bash
# Run the data generation script
python src/1_generate_data.py
```

**Expected Output:**
```
==============================================================
PNC BANKING DATA GENERATION
==============================================================

Created data/ directory
Generating 5000 customers...
✓ Generated 5000 customers
Generating accounts...
✓ Generated 8247 accounts
Generating loans...
✓ Generated 2003 loans
Generating churn data...
✓ Generated churn data for 5000 customers
  - Churned customers: 1234

Saving data to CSV files...
✓ Saved customers.csv
✓ Saved accounts.csv
✓ Saved loans.csv
✓ Saved churn.csv

==============================================================
DATA GENERATION COMPLETE!
==============================================================

==============================================================
LOADING DATA INTO POSTGRESQL DATABASE
==============================================================
Connecting to PostgreSQL database...
✓ Connected successfully
...
✓ Successfully loaded 5000 customers into database
✓ Successfully loaded 8247 accounts into database
✓ Successfully loaded 2003 loans into database
✓ Successfully loaded 5000 churn records into database

==============================================================
DATABASE LOADING COMPLETE!
==============================================================
```

### Step 7: Run Jupyter Notebook Analysis

```bash
# Start Jupyter
jupyter notebook

# OR if you prefer Jupyter Lab
jupyter lab
```

1. Navigate to `src/2_churn_analysis_and_modeling.ipynb`
2. Update database credentials in Cell 2 (if not done already)
3. Run all cells: `Cell → Run All`

### Step 8: Verify Everything Works

**Quick Database Check:**
```sql
-- Connect to database and run these queries
SELECT COUNT(*) FROM customers;  -- Should return 5000
SELECT COUNT(*) FROM accounts;   -- Should return ~8000+
SELECT COUNT(*) FROM loans;      -- Should return ~2000
SELECT COUNT(*) FROM churn;      -- Should return 5000
```

## 🔧 Troubleshooting

### Issue: "Connection refused" or "Authentication failed"

**Solution:**
1. Check PostgreSQL is running: `pg_isready`
2. Verify credentials are correct
3. Check PostgreSQL is accepting connections on localhost:5432

### Issue: "Module not found" errors

**Solution:**
```bash
# Make sure virtual environment is activated
pip install -r requirements.txt

# If still issues, try upgrading pip first
pip install --upgrade pip
```

### Issue: Data generation script fails

**Solution:**
1. Ensure the database exists: `psql -l | grep pnc_banking`
2. Ensure schema is created: Check tables exist
3. Check file permissions for creating `data/` directory

### Issue: Jupyter notebook can't find psycopg2

**Solution:**
```bash
# Install in the correct environment
pip install psycopg2-binary

# OR restart Jupyter kernel
```

## 📊 Next Steps After Setup

1. **Explore the Data:**
   - Review the generated CSV files in `data/` directory
   - Query the database directly to understand the data

2. **Analyze the Results:**
   - Review all visualizations in the notebook
   - Note the churn rate and default rate
   - Examine model performance metrics

3. **Create Tableau Dashboard:**
   - Export cleaned data from notebook if needed
   - Connect Tableau to PostgreSQL directly
   - Build visualizations based on the insights

4. **Customize the Project:**
   - Adjust the number of customers (change `NUM_CUSTOMERS`)
   - Modify churn/default probabilities in the generation script
   - Try different ML models (Random Forest, XGBoost)
   - Add more features to improve predictions

## 🎯 Project Deliverables Checklist

- [ ] Data generated and loaded into PostgreSQL
- [ ] Jupyter notebook executed successfully
- [ ] All visualizations displayed correctly
- [ ] Model trained with accuracy > 70%
- [ ] README.md updated with your findings
- [ ] Tableau dashboard created (optional but recommended)
- [ ] GitHub repository created with code
- [ ] Project added to portfolio/resume

## 💡 Tips for Portfolio Presentation

1. **Highlight Business Impact:**
   - "Identified 3 key churn drivers saving $X in retention costs"
   - "Built ML model with 85% accuracy for loan default prediction"

2. **Show Technical Skills:**
   - End-to-end pipeline (data generation → database → analysis → modeling)
   - SQL joins, Python, Machine Learning, Data Visualization
   - Database design with proper normalization

3. **Demonstrate Problem-Solving:**
   - How you handled data quality issues
   - Feature engineering decisions
   - Model selection rationale

## 📞 Need Help?

Common resources:
- PostgreSQL docs: https://www.postgresql.org/docs/
- Pandas docs: https://pandas.pydata.org/docs/
- Scikit-learn docs: https://scikit-learn.org/stable/
- Faker docs: https://faker.readthedocs.io/

---

**Good luck with your portfolio project! 🚀**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')